# 02 — Kalman Drift Filter

**Repository:** `quantum-hardware-company`  
**Directory:** `control-stack/`

This notebook upgrades `01_drift_compensation.ipynb` by replacing the moving-average estimator with a simple Kalman filter.

## Purpose

The moving-average controller stabilized drift, but it introduced lag.

This notebook asks:

> Can a state-space estimator reduce lag while preserving noise rejection?

## Workflow

1. Generate synthetic drift and calibration measurements.
2. Fit each calibration block to estimate effective parameters.
3. Compare moving-average vs Kalman drift estimates.
4. Apply compensation using each estimator.
5. Compare response-level error reduction.
6. Track CGCS / Triplet-Phase-Lock stability.
7. Save figures and a machine-readable JSON summary.

## Principle

Control follows calibration evidence.

Kalman filtering adds a state estimate:

- prediction: what the system should do next
- update: what calibration measurement says now
- control: compensate from the best current estimate


## Kalman model

We estimate drift as a one-dimensional latent state for each parameter:

$$
x_k = x_{k-1} + w_k
$$

$$
z_k = x_k + v_k
$$

where:

- $x_k$ is true latent drift,
- $z_k$ is measured drift from calibration,
- $w_k$ is process noise,
- $v_k$ is measurement noise.

The Kalman update balances prediction and measurement:

$$
\hat{x}_{k|k} = \hat{x}_{k|k-1} + K_k(z_k-\hat{x}_{k|k-1})
$$

This notebook uses independent filters for $\delta\Omega$ and $\delta B$ as a first control-stack upgrade.


In [ ]:
import os
import json

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures"
RESULTS_DIR = "results"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
def rabi_model(t, A, Omega, B):
    """Rabi oscillation probability model."""
    return A * np.sin(0.5 * Omega * t) ** 2 + B


def fit_model(model, x, y, p0, bounds=(-np.inf, np.inf)):
    """Fit model to data and return best-fit params + covariance."""
    params, cov = curve_fit(model, x, y, p0=p0, bounds=bounds, maxfev=30000)
    return params, cov


def phase_lock_metric(signal, fit):
    """Cosine similarity between observed/effective signal and target signal."""
    dot = np.dot(signal, fit)
    norm = np.linalg.norm(signal) * np.linalg.norm(fit)
    if norm == 0:
        return np.nan
    return dot / norm


def is_phase_locked(metric, threshold=1 / np.sqrt(2)):
    """CGCS threshold: cos(theta) >= 1/sqrt(2), equivalent to <= 45 degrees."""
    return metric >= threshold


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def moving_average(x, window=4):
    """Causal moving average estimate."""
    y = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i - window + 1)
        y[i] = np.mean(x[lo:i + 1])
    return y


def kalman_filter_1d(z, q, r, x0=None, p0=1.0):
    """Simple 1D random-walk Kalman filter.

    State:
        x_k = x_{k-1} + w_k
        z_k = x_k + v_k

    Args:
        z: measurements
        q: process noise variance
        r: measurement noise variance
        x0: optional initial state estimate
        p0: initial covariance

    Returns:
        x_hat: filtered state estimates
        p_hist: posterior covariance estimates
        k_hist: Kalman gains
    """
    z = np.asarray(z, dtype=float)
    x_hat = np.zeros_like(z)
    p_hist = np.zeros_like(z)
    k_hist = np.zeros_like(z)

    x = z[0] if x0 is None else float(x0)
    p = float(p0)

    for i, measurement in enumerate(z):
        # Predict
        x_pred = x
        p_pred = p + q

        # Update
        k_gain = p_pred / (p_pred + r)
        x = x_pred + k_gain * (measurement - x_pred)
        p = (1 - k_gain) * p_pred

        x_hat[i] = x
        p_hist[i] = p
        k_hist[i] = k_gain

    return x_hat, p_hist, k_hist


def evaluate_compensation(delta_omega_hat, delta_b_hat, true_delta_omega, true_delta_b):
    """Evaluate a compensation estimate against synthetic drift."""
    omega_uncomp = target_Omega + true_delta_omega
    b_uncomp = target_B + true_delta_b

    omega_comp = target_Omega + true_delta_omega - delta_omega_hat
    b_comp = np.clip(target_B + true_delta_b - delta_b_hat, 0, 0.3)

    target_signal_local = rabi_model(pulse_t, target_A, target_Omega, target_B)

    uncomp_response_rmse = []
    comp_response_rmse = []

    metric_uncomp = []
    metric_comp = []

    for k in range(len(true_delta_omega)):
        y_uncomp = rabi_model(pulse_t, target_A, omega_uncomp[k], b_uncomp[k])
        y_comp = rabi_model(pulse_t, target_A, omega_comp[k], b_comp[k])

        uncomp_response_rmse.append(rmse(y_uncomp, target_signal_local))
        comp_response_rmse.append(rmse(y_comp, target_signal_local))

        metric_uncomp.append(phase_lock_metric(y_uncomp, target_signal_local))
        metric_comp.append(phase_lock_metric(y_comp, target_signal_local))

    return {
        "omega_comp": omega_comp,
        "b_comp": b_comp,
        "omega_rmse": rmse(omega_comp, np.full(len(omega_comp), target_Omega)),
        "b_rmse": rmse(b_comp, np.full(len(b_comp), target_B)),
        "response_rmse": np.array(comp_response_rmse),
        "response_rmse_mean": float(np.mean(comp_response_rmse)),
        "phase_lock_metric": np.array(metric_comp),
        "min_phase_lock": float(np.min(metric_comp)),
        "mean_phase_lock": float(np.mean(metric_comp)),
        "max_angle_degrees": float(np.max(np.degrees(np.arccos(np.clip(metric_comp, -1, 1))))),
        "uncomp_response_rmse": np.array(uncomp_response_rmse),
        "uncomp_phase_lock_metric": np.array(metric_uncomp),
    }


## Generate target behavior and uncompensated drift

We use the same control-stack setup as notebook 01 so this notebook is directly comparable.


In [ ]:
# Calibration blocks / lab-time index
n_blocks = 36
block_times = np.arange(n_blocks)

# Pulse duration axis inside each calibration block
n_points_per_block = 140
pulse_t = np.linspace(0, 10, n_points_per_block)

# Target parameters
target_A = 0.88
target_Omega = 2.45
target_B = 0.06

target_signal = rabi_model(pulse_t, target_A, target_Omega, target_B)

# True environmental drift
true_delta_Omega = (
    0.055 * np.sin(2 * np.pi * block_times / 21 + 0.4)
    + 0.018 * np.sin(2 * np.pi * block_times / 9)
)
true_delta_B = (
    0.0016 * block_times
    + 0.006 * np.sin(2 * np.pi * block_times / 14 + 0.2)
)

# Uncompensated effective parameters
Omega_uncomp = target_Omega + true_delta_Omega
B_uncomp = target_B + true_delta_B
A_uncomp = target_A + 0.012 * np.sin(2 * np.pi * block_times / 17)

print(f"Target Omega = {target_Omega:.4f}")
print(f"Target B     = {target_B:.4f}")
print(f"Uncompensated Omega range = {Omega_uncomp.min():.4f} to {Omega_uncomp.max():.4f}")
print(f"Uncompensated B range     = {B_uncomp.min():.4f} to {B_uncomp.max():.4f}")


## Simulate calibration measurements and estimate drift

Each block is measured and fit independently.

The fitted parameter deviation becomes the measurement input to each estimator.


In [ ]:
Y_uncomp_obs = []
Y_uncomp_clean = []

for k in range(n_blocks):
    y_clean = rabi_model(pulse_t, A_uncomp[k], Omega_uncomp[k], B_uncomp[k])
    noise = 0.04 * np.random.randn(n_points_per_block)
    y_obs = np.clip(y_clean + noise, 0, 1)
    Y_uncomp_clean.append(y_clean)
    Y_uncomp_obs.append(y_obs)

Y_uncomp_clean = np.array(Y_uncomp_clean)
Y_uncomp_obs = np.array(Y_uncomp_obs)

fit_params = []
fit_stds = []
fit_curves = []

initial_guess = [0.85, 2.40, 0.05]
bounds = ([0.0, 0.0, 0.0], [1.2, 5.0, 0.3])

for k in range(n_blocks):
    params, cov = fit_model(
        rabi_model,
        pulse_t,
        Y_uncomp_obs[k],
        p0=initial_guess,
        bounds=bounds,
    )
    fit_params.append(params)
    fit_stds.append(np.sqrt(np.diag(cov)))
    fit_curves.append(rabi_model(pulse_t, *params))

fit_params = np.array(fit_params)
fit_stds = np.array(fit_stds)
fit_curves = np.array(fit_curves)

A_est = fit_params[:, 0]
Omega_est = fit_params[:, 1]
B_est = fit_params[:, 2]

delta_Omega_est = Omega_est - target_Omega
delta_B_est = B_est - target_B

print("Calibration fit complete.")
print(f"Mean estimated Omega drift = {np.mean(delta_Omega_est):.6f}")
print(f"Mean estimated B drift     = {np.mean(delta_B_est):.6f}")


## Baseline estimator: moving average

This reproduces the notebook-01 baseline.


In [ ]:
ma_window = 4

delta_Omega_ma = moving_average(delta_Omega_est, window=ma_window)
delta_B_ma = moving_average(delta_B_est, window=ma_window)

ma_eval = evaluate_compensation(
    delta_Omega_ma,
    delta_B_ma,
    true_delta_Omega,
    true_delta_B,
)

print(f"Moving-average window = {ma_window}")
print(f"MA Omega RMSE = {ma_eval['omega_rmse']:.6f}")
print(f"MA B RMSE     = {ma_eval['b_rmse']:.6f}")
print(f"MA response RMSE mean = {ma_eval['response_rmse_mean']:.6f}")


## Kalman estimator

The Kalman filter needs two tuning values:

- `q`: process-noise variance — how quickly drift is allowed to change
- `r`: measurement-noise variance — how noisy calibration estimates are

We estimate measurement variance from block-fit uncertainty, then choose process variance to allow smoother but responsive tracking.


In [ ]:
# Measurement variance estimates from fit uncertainty
r_omega = float(np.median(fit_stds[:, 1] ** 2))
r_b = float(np.median(fit_stds[:, 2] ** 2))

# Process variances tuned for this synthetic drift profile.
# Larger q = more responsive, less smoothing.
q_omega = 8.0e-4
q_b = 1.0e-5

delta_Omega_kf, p_Omega_hist, k_Omega_hist = kalman_filter_1d(
    delta_Omega_est,
    q=q_omega,
    r=r_omega,
    p0=1.0,
)

delta_B_kf, p_B_hist, k_B_hist = kalman_filter_1d(
    delta_B_est,
    q=q_b,
    r=r_b,
    p0=1.0,
)

kf_eval = evaluate_compensation(
    delta_Omega_kf,
    delta_B_kf,
    true_delta_Omega,
    true_delta_B,
)

print("Kalman filter tuning:")
print(f"q_omega = {q_omega:.8f}, r_omega = {r_omega:.8f}")
print(f"q_b     = {q_b:.8f}, r_b     = {r_b:.8f}")
print()
print(f"KF Omega RMSE = {kf_eval['omega_rmse']:.6f}")
print(f"KF B RMSE     = {kf_eval['b_rmse']:.6f}")
print(f"KF response RMSE mean = {kf_eval['response_rmse_mean']:.6f}")


## Drift-estimator comparison

Kalman filtering should track the true drift with less lag than the moving average while still suppressing measurement noise.


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(block_times, true_delta_Omega, linewidth=2, label="true Omega drift")
plt.plot(block_times, delta_Omega_est, marker="o", linewidth=1, label="measured Omega drift")
plt.plot(block_times, delta_Omega_ma, linewidth=2, linestyle="--", label="moving average")
plt.plot(block_times, delta_Omega_kf, linewidth=2, linestyle=":", label="Kalman filter")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("Omega drift")
plt.title("Estimator comparison: Rabi frequency drift")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/kalman_01_omega_estimator_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/kalman_01_omega_estimator_comparison.pdf")

plt.show()


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(block_times, true_delta_B, linewidth=2, label="true offset drift")
plt.plot(block_times, delta_B_est, marker="o", linewidth=1, label="measured offset drift")
plt.plot(block_times, delta_B_ma, linewidth=2, linestyle="--", label="moving average")
plt.plot(block_times, delta_B_kf, linewidth=2, linestyle=":", label="Kalman filter")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("readout offset drift")
plt.title("Estimator comparison: readout offset drift")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/kalman_02_offset_estimator_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/kalman_02_offset_estimator_comparison.pdf")

plt.show()


## Kalman gain diagnostics

Kalman gain shows how much the filter trusts each new calibration measurement.

Higher gain means stronger update from measurement.  
Lower gain means stronger trust in model prediction.


In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(block_times, k_Omega_hist, marker="o", linewidth=1, label="Omega Kalman gain")
plt.plot(block_times, k_B_hist, marker="o", linewidth=1, label="B Kalman gain")
plt.xlabel("calibration block / lab time index")
plt.ylabel("Kalman gain")
plt.title("Kalman gain over calibration blocks")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/kalman_03_gain_trace.png", dpi=300)
plt.savefig(f"{FIG_DIR}/kalman_03_gain_trace.pdf")

plt.show()


## Compensation comparison: parameter error

We compare three cases:

1. no compensation
2. moving-average compensation
3. Kalman compensation


In [ ]:
Omega_error_uncomp = Omega_uncomp - target_Omega
Omega_error_ma = ma_eval["omega_comp"] - target_Omega
Omega_error_kf = kf_eval["omega_comp"] - target_Omega

plt.figure(figsize=(9, 5))
plt.plot(block_times, Omega_error_uncomp, marker="o", linewidth=1, label="uncompensated")
plt.plot(block_times, Omega_error_ma, marker="o", linewidth=1, label="moving average")
plt.plot(block_times, Omega_error_kf, marker="o", linewidth=1, label="Kalman")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("Omega error from target")
plt.title("Compensation comparison: Rabi frequency error")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/kalman_04_omega_error_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/kalman_04_omega_error_comparison.pdf")

plt.show()


In [ ]:
B_error_uncomp = B_uncomp - target_B
B_error_ma = ma_eval["b_comp"] - target_B
B_error_kf = kf_eval["b_comp"] - target_B

plt.figure(figsize=(9, 5))
plt.plot(block_times, B_error_uncomp, marker="o", linewidth=1, label="uncompensated")
plt.plot(block_times, B_error_ma, marker="o", linewidth=1, label="moving average")
plt.plot(block_times, B_error_kf, marker="o", linewidth=1, label="Kalman")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("offset error from target")
plt.title("Compensation comparison: readout offset error")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/kalman_05_offset_error_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/kalman_05_offset_error_comparison.pdf")

plt.show()


## Response-level comparison

This is the core result: how close the controlled response stays to target behavior.


In [ ]:
uncomp_response_rmse = ma_eval["uncomp_response_rmse"]

plt.figure(figsize=(9, 5))
plt.plot(block_times, uncomp_response_rmse, marker="o", linewidth=1, label="uncompensated")
plt.plot(block_times, ma_eval["response_rmse"], marker="o", linewidth=1, label="moving average")
plt.plot(block_times, kf_eval["response_rmse"], marker="o", linewidth=1, label="Kalman")
plt.xlabel("calibration block / lab time index")
plt.ylabel("RMSE vs target response")
plt.title("Response-level error: moving average vs Kalman control")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/kalman_06_response_error_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/kalman_06_response_error_comparison.pdf")

plt.show()


## Example block comparison

We choose the block where the moving-average controller has its largest response error and compare against Kalman compensation.


In [ ]:
example_block = int(np.argmax(ma_eval["response_rmse"]))

target_signal = rabi_model(pulse_t, target_A, target_Omega, target_B)
y_uncomp = rabi_model(pulse_t, target_A, Omega_uncomp[example_block], B_uncomp[example_block])
y_ma = rabi_model(pulse_t, target_A, ma_eval["omega_comp"][example_block], ma_eval["b_comp"][example_block])
y_kf = rabi_model(pulse_t, target_A, kf_eval["omega_comp"][example_block], kf_eval["b_comp"][example_block])

plt.figure(figsize=(9, 5))
plt.plot(pulse_t, target_signal, linewidth=2, label="target response")
plt.plot(pulse_t, y_uncomp, linewidth=2, label="uncompensated")
plt.plot(pulse_t, y_ma, linewidth=2, label="moving average")
plt.plot(pulse_t, y_kf, linewidth=2, label="Kalman")
plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.title(f"Control comparison for block {example_block}")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/kalman_07_example_block_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/kalman_07_example_block_comparison.pdf")

plt.show()


## CGCS stability comparison

We compare phase-lock margin for:

- uncompensated drift
- moving-average compensation
- Kalman compensation


In [ ]:
threshold = 1 / np.sqrt(2)

metric_uncomp = ma_eval["uncomp_phase_lock_metric"]
metric_ma = ma_eval["phase_lock_metric"]
metric_kf = kf_eval["phase_lock_metric"]

plt.figure(figsize=(9, 5))
plt.axhline(threshold, linewidth=1, linestyle="--", label="45° threshold")
plt.plot(block_times, metric_uncomp, marker="o", linewidth=1, label="uncompensated")
plt.plot(block_times, metric_ma, marker="o", linewidth=1, label="moving average")
plt.plot(block_times, metric_kf, marker="o", linewidth=1, label="Kalman")
plt.ylim(0.68, 1.01)
plt.xlabel("calibration block / lab time index")
plt.ylabel("cosine similarity vs target")
plt.title("CGCS stability: moving average vs Kalman control")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/kalman_08_cgcs_stability_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/kalman_08_cgcs_stability_comparison.pdf")

plt.show()


## Compact improvement summary

This compares error reduction from the uncompensated baseline.


In [ ]:
Omega_rmse_uncomp = rmse(Omega_uncomp, np.full(n_blocks, target_Omega))
B_rmse_uncomp = rmse(B_uncomp, np.full(n_blocks, target_B))
response_rmse_uncomp_mean = float(np.mean(uncomp_response_rmse))

ma_omega_improvement = 1 - ma_eval["omega_rmse"] / Omega_rmse_uncomp
ma_b_improvement = 1 - ma_eval["b_rmse"] / B_rmse_uncomp
ma_response_improvement = 1 - ma_eval["response_rmse_mean"] / response_rmse_uncomp_mean

kf_omega_improvement = 1 - kf_eval["omega_rmse"] / Omega_rmse_uncomp
kf_b_improvement = 1 - kf_eval["b_rmse"] / B_rmse_uncomp
kf_response_improvement = 1 - kf_eval["response_rmse_mean"] / response_rmse_uncomp_mean

labels = ["Omega error", "Offset error", "Response error"]
ma_values = [ma_omega_improvement, ma_b_improvement, ma_response_improvement]
kf_values = [kf_omega_improvement, kf_b_improvement, kf_response_improvement]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(9, 5))
bars_ma = plt.bar(x - width / 2, ma_values, width, label="moving average")
bars_kf = plt.bar(x + width / 2, kf_values, width, label="Kalman")

plt.xticks(x, labels)
plt.ylim(0, 1)
plt.ylabel("error reduction fraction")
plt.title("Control improvement: moving average vs Kalman")
plt.legend()

for bars, values in [(bars_ma, ma_values), (bars_kf, kf_values)]:
    for bar, value in zip(bars, values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.025,
            f"{100 * value:.1f}%",
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()

plt.savefig(f"{FIG_DIR}/kalman_09_improvement_summary.png", dpi=300)
plt.savefig(f"{FIG_DIR}/kalman_09_improvement_summary.pdf")

plt.show()


## Kalman tuning sweep

Kalman performance depends on process variance $q$.

This sweep tests a range of `q_omega` values while holding measurement variance fixed.


In [ ]:
q_omega_values = np.logspace(-6, -2, 25)
sweep_response_rmse = []
sweep_omega_rmse = []

for q_test in q_omega_values:
    omega_kf_test, _, _ = kalman_filter_1d(delta_Omega_est, q=q_test, r=r_omega, p0=1.0)
    eval_test = evaluate_compensation(
        omega_kf_test,
        delta_B_kf,
        true_delta_Omega,
        true_delta_B,
    )
    sweep_response_rmse.append(eval_test["response_rmse_mean"])
    sweep_omega_rmse.append(eval_test["omega_rmse"])

sweep_response_rmse = np.array(sweep_response_rmse)
sweep_omega_rmse = np.array(sweep_omega_rmse)
best_q_idx = int(np.argmin(sweep_response_rmse))
best_q_omega = float(q_omega_values[best_q_idx])

print(f"Current q_omega = {q_omega:.8f}")
print(f"Best swept q_omega = {best_q_omega:.8f}")
print(f"Best swept response RMSE = {sweep_response_rmse[best_q_idx]:.6f}")


In [ ]:
plt.figure(figsize=(9, 5))
plt.semilogx(q_omega_values, sweep_response_rmse, marker="o", linewidth=1)
plt.axvline(q_omega, linewidth=1, linestyle="--", label=f"current q = {q_omega:.1e}")
plt.axvline(best_q_omega, linewidth=1, linestyle=":", label=f"best q = {best_q_omega:.1e}")
plt.xlabel("Omega process variance q")
plt.ylabel("mean compensated response RMSE")
plt.title("Kalman tuning sweep: Omega process variance")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/kalman_10_q_sweep_response.png", dpi=300)
plt.savefig(f"{FIG_DIR}/kalman_10_q_sweep_response.pdf")

plt.show()


## Save control-stack summary


In [ ]:
summary = {
    "n_blocks": int(n_blocks),
    "moving_average_window": int(ma_window),
    "kalman_q_omega": float(q_omega),
    "kalman_r_omega": float(r_omega),
    "kalman_q_b": float(q_b),
    "kalman_r_b": float(r_b),
    "Omega_rmse_uncompensated": float(Omega_rmse_uncomp),
    "Omega_rmse_moving_average": float(ma_eval["omega_rmse"]),
    "Omega_rmse_kalman": float(kf_eval["omega_rmse"]),
    "B_rmse_uncompensated": float(B_rmse_uncomp),
    "B_rmse_moving_average": float(ma_eval["b_rmse"]),
    "B_rmse_kalman": float(kf_eval["b_rmse"]),
    "response_rmse_uncompensated_mean": float(response_rmse_uncomp_mean),
    "response_rmse_moving_average_mean": float(ma_eval["response_rmse_mean"]),
    "response_rmse_kalman_mean": float(kf_eval["response_rmse_mean"]),
    "Omega_error_reduction_moving_average_fraction": float(ma_omega_improvement),
    "Omega_error_reduction_kalman_fraction": float(kf_omega_improvement),
    "B_error_reduction_moving_average_fraction": float(ma_b_improvement),
    "B_error_reduction_kalman_fraction": float(kf_b_improvement),
    "response_error_reduction_moving_average_fraction": float(ma_response_improvement),
    "response_error_reduction_kalman_fraction": float(kf_response_improvement),
    "min_phase_lock_uncompensated": float(np.min(metric_uncomp)),
    "min_phase_lock_moving_average": float(np.min(metric_ma)),
    "min_phase_lock_kalman": float(np.min(metric_kf)),
    "mean_phase_lock_uncompensated": float(np.mean(metric_uncomp)),
    "mean_phase_lock_moving_average": float(np.mean(metric_ma)),
    "mean_phase_lock_kalman": float(np.mean(metric_kf)),
    "phase_lock_threshold": float(threshold),
    "all_kalman_blocks_phase_locked": bool(np.all(metric_kf >= threshold)),
    "best_swept_q_omega": float(best_q_omega),
    "best_swept_q_omega_response_rmse": float(sweep_response_rmse[best_q_idx]),
}

summary


In [ ]:
with open(f"{RESULTS_DIR}/kalman_drift_filter_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved {RESULTS_DIR}/kalman_drift_filter_summary.json")


## Zip outputs for download from Colab

Run this cell after all figures/results have been generated.


In [ ]:
!zip -r kalman_drift_filter_outputs.zip figures results


## Control-stack status

This notebook upgrades the control stack:

```text
moving-average drift compensation → Kalman drift filtering
```

## Interpretation

The Kalman filter provides a state-space estimator that can reduce lag and tune responsiveness using process variance.

## Next steps

Possible next notebooks:

- `03_control_policy_comparison.ipynb` — compare no control, moving average, Kalman, and predictive control.
- `04_joint_parameter_filter.ipynb` — estimate Ω and B jointly with covariance.
- `noise-mitigation/01_structured_drift_model.ipynb` — model remaining compensated residuals as noise processes.

Guiding rule:

> Control follows calibration evidence.
